# attributes

- `df.index`
- `df.columns`
- `df.shape[0]` # of rows; `df.shape[1]` # of columns
- `df.info()`
- `df.describe()`

In [1]:
from IPython.display import display
import numpy as np
import pandas as pd

df = pd.DataFrame({'age':[15, 17, 10],'growth':[.5, .7, 1.2],'Name':['Paul', 'George', 'Ringo'] })
display(df.columns)
display(df.index)
display(df.info())


df0 = pd.DataFrame({
    "firm":    ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
    "year":    [2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023],
    "region":  ["East", "East", "East", "East", "West", "West", "West", "West", "East", "East", "East", "East"],
    "sales":   [120, 135, 135, 160, 90, 110, 105, 140, 120, np.nan, 130, 130],
    "profit":  [15, 18, 18, 25, 10, 12, 11, 20, 15, 16, 19, 19]
}).rename(index={i: f'id-{i}' for i in range(12) })

df0

Index(['age', 'growth', 'Name'], dtype='str')

RangeIndex(start=0, stop=3, step=1)

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     3 non-null      int64  
 1   growth  3 non-null      float64
 2   Name    3 non-null      str    
dtypes: float64(1), int64(1), str(1)
memory usage: 219.0 bytes


None

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-5,B,2021,West,110.0,12
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16


Note: `apply()` for series, but `.pipe()` or `.assign()` apply functions to the whole DataFrame.

In [ ]:
display(df.sum(axis=0))
display(df.iloc[:,:2].sum(axis=1)) 

age       42.0
growth     2.4
dtype: float64

0    15.5
1    17.7
2    11.2
dtype: float64

In [ ]:
# sum over rows
display(df.iloc[:,:2].apply(np.sum, axis=0))
# sum over columns
display(df.iloc[:,:2].apply(np.sum, axis='columns'))

age       42.0
growth     2.4
dtype: float64

0    15.5
1    17.7
2    11.2
dtype: float64

In [8]:
(df
 .rename(columns={'Age': 'New_Age'})
 .assign(Name=lambda df_:df_.Name.str.lower())
)

,age,growth,Name
0,15,0.5,paul
1,17,0.7,george
2,10,1.2,ringo


In [9]:
snow=pd.DataFrame(
    {'Obs Date': ['1980/01/01', '1980/01/02'],
    'Precip':['0.1', 'T'],
    'Snowfall': [1,0],
    'T.Obs':[25, 18]})

snow.rename(columns=lambda c: c.lower().replace(' ', '_').replace('.', '_'))\
    .assign(
        obs_date=lambda df2: pd.to_datetime(df2['obs_date']),
        precip=lambda df3: df3['precip'].str.replace("T", '0').astype('float')
    )

,obs_date,precip,snowfall,t_obs
0,1980-01-01,0.1,1,25
1,1980-01-02,0.0,0,18


# add

- need to align index

| Method | Description |
|---|---|
| .add(other, axis='columns', fill_value=None) | Add `other` to DataFrame across the axis.|
| .sub(other, axis='columns', fill_value=None) | Subtract `other` from DataFrame across the axis. |
| .mul(other, axis='columns', fill_value=None) | Multiply `other` with DataFrame across the axis. |
| .div(other, axis='columns', fill_value=None) | Divide DataFrame by `other` across the axis.  |
| .min(axis=None, skipna=None, numeric_only=None) | Return minimum value of each column. |
| .max(axis=None, skipna=None, numeric_only=None) | Return maximum value of each column. |
| .mean(axis=None, skipna=None, numeric_only=None) | Return mean value of each column. |


In [26]:
(df0.loc[:,['profit']]
 .reset_index(drop=True)
 .add(np.arange(df0.shape[0])
      .reshape(-1,1)*5)
).head(2)

,profit
0,15
1,23


# loop
- `for col_name, col_series in df.items():` over columns
- `for indx, row_series in df.iterrows():` over rows

In [29]:
for idx, row in df.iterrows():
    print(f"{idx}, and {row['Name']}")

0, and Paul
1, and George
2, and Ringo


# aggregate

- using directly
- using within `agg()` in 3 ways:
    - `['count','size', my_custom_func]` for all columns
    - `df.agg({'a': ['count', 'sum'], 'b': ['min']})` for customized func each column
    - `.groupby('my_group').agg(non_missing_count=('old_var', 'count'))`

| Method | Description |
|---|---|
| .sum(axis=0, skipna=True, level=None, numeric_only=None, min_count=0) | Return sum over the axis. Default of empty sequence is 0. Set `min_count=1` to return `NaN`. |
| .min(axis=0, skipna=True, level=None, numeric_only=None) | Return minimum over the axis. |
| .max(axis=0, skipna=True, level=None, numeric_only=None) | Return maximum over the axis. |
| .idxmin(axis=0, skipna=True) | Return the index of the first minimum value over the axis. |
| .idxmax(axis=0, skipna=True) | Return the index of the first maximum value over the axis. |
| .agg(func=None, axis=0, *args, **kwargs) | Aggregate using `func` over the axis. `func` can be a function, string, list/dict of functions/strings. |

In [39]:

def return_col_number(tab):
    return (df.columns.get_loc(tab.name))

display(df.select_dtypes('number').agg(['count', 'sum','size', return_col_number], axis=0)
        .rename(index={'return_col_number':'Col Position'}))

display(df.agg({'age':['max','min'], 'growth':['mean']}).fillna(0))

display(df.agg(lambda col: col.loc[1]))

,age,growth
count,3,3.0
sum,42,2.4
size,3,3.0
Col Position,0,1.0


,age,growth
max,17.0,0.0
min,10.0,0.0
mean,0.0,0.8


age           17
growth       0.7
Name      George
dtype: object

# apply

In [ ]:
(df.select_dtypes('number')
 .apply(lambda row: row.max()-row.min(), axis='columns')
    .rename('range') 
 )

0    14.5
1    16.3
2     8.8
Name: range, dtype: float64

In [2]:
df.select_dtypes('number')\
    .pipe(lambda df_:df_.max(axis='columns') - df_.min(axis='columns'))\
    .rename('range')

0    14.5
1    16.3
2     8.8
Name: range, dtype: float64

# assign column type 

| Method | Description |
|---|---|
| `astype(dtype, copy=True, errors='raise')` or  `.astype({'<column name>': '<data type>'})` | Cast dataframe into `dtype`.|
| `assign(**kwargs)` | Return a new DataFrame with updated or new columns. `kwargs` maps column name to function, scalar, or Series. If using a function, it is passed the current state of the DataFrame and should return a scalar or Series. Subsequent columns may reference earlier columns in `kwargs` if you use a function. |

In [6]:
df0.select_dtypes('float64').columns

Index(['sales'], dtype='str')

In [9]:
(df0.astype({col: 'float32' for col in df0.select_dtypes('float64').columns}).head(3))

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18


In [17]:
# creates a new column called Average_rank by doing this: 
# Add sales and profit each row to get one total score per row.
# Rank the rows by that total score. Use dense ranking (same value, same rank, not skip)
(
df0
.assign(Average_rank=(lambda df_:df_[['sales', 'profit']].agg('sum', axis=1)
                      .rank(method='dense') ))
                      .head(3)
)


,firm,year,region,sales,profit,Average_rank
id-0,A,2020,East,120.0,15,5.0
id-1,A,2021,East,135.0,18,7.0
id-2,A,2022,East,135.0,18,7.0


In [15]:
# the lambda function in the assign uses the series as the 
# input and return a series. But it also works by returning a scalar and broadcast to a new column. 
(df0
 .assign(Median_rank=
               (lambda df_:df_[['sales', 'profit']].agg('sum', axis=1)\
.rank(method='dense')
.quantile(0.5)))
.head(3)
)

,firm,year,region,sales,profit,Median_rank
id-0,A,2020,East,120.0,15,5.5
id-1,A,2021,East,135.0,18,5.5
id-2,A,2022,East,135.0,18,5.5


# create/update columns

- `assign()` is the most versatile!

| Method | Description |
|---|---|
| `rename(mapper=None, index=None, columns=None, axis=0, copy=True, level=None, errors='ignore')` | Change axis labels. Pass `columns` or `index` as a dictionary (mapping old values to new values) or a function (accepting the old value and returning the new value). |
| `replace(to_replace=None, value=None, limit=None, regex=False, method='pad')` | Replace values from `to_replace` (string, regex, number, Series, list, or dict mapping replacements) with `value`. If `to_replace` is a list and `value` is `None`, you can fill using `method='bfill'` or `'ffill'`. |
| `drop(labels=None, axis=0, index=None, columns=None, level=None, errors='raise')` | Drop rows or columns with specified labels. Prefer `columns='age'` rather than `labels='age'` with `axis=1`. |
| `dropna(axis=0, subset=['var 1'])` | Drop rows (`axis=0`) or columns (`axis=1`) (may use `replace('', pd.NA)` first)|
| `query(expr)` | Evaluate `expr` to filter the DataFrame. Refer to Python variables by prefixing with `@`. Use backticks around column names that contain spaces or special characters. |
| `assign(**kwargs)` | Return a new DataFrame with updated or new columns. `kwargs` maps column name to a function, scalar, or Series. If using a function, it is passed the current state of the DataFrame and should return a scalar or Series. Subsequent columns may reference earlier `kwargs` columns when functions are used. |

In [18]:
df.rename(columns=lambda c: c.replace('a', 'A'))

,Age,growth,NAme
0,15,0.5,Paul
1,17,0.7,George
2,10,1.2,Ringo


In [19]:
df0.assign(sales=lambda df_:df_.sales.fillna(0))

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-5,B,2021,West,110.0,12
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,0.0,16


# fill missings/dedupe

- `.dropna`
- `.fillna`
- `.interpolate`
- `value_counts()` to know which ones are duplicates
- `duplicated(keep='')` return boolean of duplicates
- check if missing values are present
    - check the proportion of missing values
        - impute data
        - or delete rows with missing values

| Method | Description |
|---|---|
| `.isna()` | Return a boolean DataFrame with the same dimensions with `True` where cells are missing. |
| `.sum(axis=0, skipna=True, level=None, numeric_only=None, min_count=0)` | Return sum over the axis. The default for an empty sequence is `0`; set `min_count=1` to require at least one non-NA value (otherwise returns `NaN`). |
| `.mean(axis=0, skipna=True, level=None, numeric_only=None)` | Return mean over the axis (ignores NA by default when `skipna=True`). |
| `.drop_duplicates(subset=None, keep='first', ignore_index=False)` | Return a DataFrame with duplicated rows removed. Use `subset` to consider specific columns. `keep` may be `'first'`, `'last'`, or `False` (drop all duplicates). Set `ignore_index=True` to reset the index. |

In [20]:
df0.sales.isna().sum()

np.int64(1)

In [29]:
df0.query('sales.isna()')

,firm,year,region,sales,profit
id-9,C,2021,East,NaN,16


# sort columns and indexes

| Method | Description |
|---|---|
| `.sort_values(by, axis=0, ascending=True, kind='quicksort', na_position='last', ignore_index=False, key=None)` | Return dataframe with values sorted along the axis. Use `by` to specify a column (string) or a list of columns (for `axis=0`). You can use `kind='mergesort'` or `kind='stable'` for a stable sort if only sorting one column. A `key` function accepts a `Series` and should return a `Series` with the same index. |
| `.sort_index(axis=0, level=None, ascending=True, kind='quicksort', na_position='last', sort_remaining=True, ignore_index=False, key=None)` | Return dataframe with index (`axis=0`) or columns (`axis=1`) sorted. Can specify a single level or multiple levels with `level`. Can specify the column (string) or a list of columns (for `axis=0`). Can use `kind='mergesort'` or `kind='stable'` for a stable sort if only sorting one column. You can reset the index with `ignore_index`. A `key` function accepts an index and should return an index. For multi-level indexes, each index is passed in independently to the function. |

In [4]:
display(df0.sort_values(by=['profit', 'sales'],ascending=[True, False]).head(3))

,firm,year,region,sales,profit
id-4,B,2020,West,90.0,10
id-6,B,2022,West,105.0,11
id-5,B,2021,West,110.0,12


In [5]:
# putting above into the key function for the sort
(df0.sort_values(by='region',
    key=lambda name_ser: name_ser.astype(str).str.len())
)

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-4,B,2020,West,90.0,10
id-5,B,2021,West,110.0,12
id-6,B,2022,West,105.0,11
id-7,B,2023,West,140.0,20
id-8,C,2020,East,120.0,15
id-9,C,2021,East,NaN,16


In [ ]:
# sort columns
df0.sort_index(axis='columns').head(3)

,firm,profit,region,sales,year
id-0,A,15,East,120.0,2020
id-1,A,18,East,135.0,2021
id-2,A,18,East,135.0,2022


In [ ]:
# sort column based on the length of names
df0.sort_index(axis='columns', ascending=False, key=lambda name_ser: name_ser.str.len() ).head(3)


,region,profit,sales,firm,year
id-0,East,15,120.0,A,2020
id-1,East,18,135.0,A,2021
id-2,East,18,135.0,A,2022


Sorting the index allows us to slice the index by name.
| Method | Description |
|---|---|
| `.set_index(keys, drop=True, append=False, verify_integrity=False)` | Return dataframe with the new index. The `keys` argument can be a column name, a `Series` (or numpy array) of labels for the index, or a list of column names or series. The `drop` parameter indicates whether to remove columns used for the index. The `append` parameter allows you to add additional index levels. You can check for duplicate index values by setting `verify_integrity=True`. |
| `.loc` | Attribute to index off of by index and column names. Slices use the closed interval (including start and end). |


In [19]:
(
df0
.set_index('profit')
.sort_index().loc[15:20, 'firm': 'region']
 )

,firm,year,region
profit,,,
15,A,2020,East
15,C,2020,East
16,C,2021,East
18,A,2021,East
18,A,2022,East
19,C,2022,East
19,C,2023,East
20,B,2023,West


In [22]:
df0[df0.profit>15]

,firm,year,region,sales,profit
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18
id-3,A,2023,East,160.0,25
id-7,B,2023,West,140.0,20
id-9,C,2021,East,NaN,16
id-10,C,2022,East,130.0,19
id-11,C,2023,East,130.0,19


In [27]:
df0['sales'].hasnans

True

In [39]:
column_missing=[]
for i in df0.columns:
         if df0[i].hasnans:
            column_missing.append(i)
df0.loc[:,column_missing].head(2)

,sales
id-0,120.0
id-1,135.0


In [ ]:
df0.isna().any()

firm      False
year      False
region    False
sales     False
profit    False
dtype: bool

In [38]:
df0.loc[:, df0.isna().any()].head(2)

,sales
id-0,120.0
id-1,135.0


In [44]:
df0[df0['sales']>10]
df0.loc[df0['sales']>10]
df0.iloc[:, 5:10]
df0.iloc[:, [1,3]]
df0.loc[:,['region','sales']]
df0.loc[:, 'region':'sales']
df0.loc[:, df0.isna().any()]
df0.iloc[ [ i for i in range(10) if i%2==0], :]

,firm,year,region,sales,profit
id-0,A,2020,East,120.0,15
id-2,A,2022,East,135.0,18
id-4,B,2020,West,90.0,10
id-6,B,2022,West,105.0,11
id-8,C,2020,East,120.0,15


# filter/index

## rename index
| Method | Description |
|---|---|
| `.rename(mapper=None, index=None, columns=None, axis=0, copy=True, level=None, errors='ignore')` | Change axis labels. Pass the `columns` or `index` as a dictionary (mapping old values to new values) or a function (accepting the old value and returning the new value). |

In [48]:
def name_to_initial(val):
    names = val.split()
    return ' '.join([f'{names[0][0]}.', '-Region'])

df0.set_index('region').rename(index=name_to_initial).head(3)

,firm,year,sales,profit
region,,,,
E. -Region,A,2020,120.0,15
E. -Region,A,2021,135.0,18
E. -Region,A,2022,135.0,18


In [54]:
def name_to_upper(val):
    return (val[0:3]+'.').upper()

df0.rename(columns=name_to_upper).head(3)

,FIR.,YEA.,REG.,SAL.,PRO.
id-0,A,2020,East,120.0,15
id-1,A,2021,East,135.0,18
id-2,A,2022,East,135.0,18


## reset the Index

To a monotonically increase the integer index for a dataframe, use the `.reset_index` method.

| Method | Description |
|---|---|
| `.reset_index(level=None, drop=False, col_level=0, col_fill='')` | Return a dataframe with the new index (or new level). To remove a level, specify that with `level` (by position or name). Position `0` is the outermost level, and it goes up. Alternatively, `-1` is the innermost level. Index values are moved to columns or dropped if `drop=True`. `col_level` determines where the index label goes with multiple column levels. Other levels will get the value of `col_fill`. |

In [ ]:
# make the region the first column
df0.set_index('region').reset_index().head(3)

,region,firm,year,sales,profit
0,East,A,2020,120.0,15
1,East,A,2021,135.0,18
2,East,A,2022,135.0,18


## Dataframe Indexing, Filtering, & Querying
| Method | Description |
|---|---|
| `.set_index(keys, drop=True, append=False, verify_integrity=False)` | Return a dataframe with a new index. The `keys` argument can be a column name, a `Series` (or numpy array) of labels for the index, or a list of column names or series. The `drop` parameter indicates whether to remove columns used for the index. The `append` parameter allows you to add additional index levels. You can check for duplicate index values by setting `verify_integrity=True`. |
| `.sort_index(axis=0, level=None, ascending=True, kind='quicksort', na_position='last', sort_remaining=True, ignore_index=False, key=None)` | Return a dataframe with the index (`axis=0`) or columns (`axis=1`) sorted. Can specify a single level or multiple levels with `level`. Can specify the direction of each level sort with `ascending`. Choose the axis (default is `axis=0`). Use `by` to specify a column (string) or a list of columns (for `axis=0`). Can use `kind='mergesort'` or `kind='stable'` for a stable sort if only sorting one column. Can reset the index with `ignore_index`. A `key` function accepts an index and should return an index. For multi-level indexes, each index is passed in independently to the function. |
| `.query(expr)` | Evaluate `expr` to filter the dataframe. Refer to variables by prefixing them with `@`. Use backticks around the column names with spaces. |

- `query('var.isna()')`, `query('var.notna()')`
- `query('var>10')`
- `query('var!="text"')`
- `query('var.isin(@list_name)')` , `query('var in @list_name')`, `query('var not in @list_name')`
- **and**, **or**, **not** 
- **==** for equality

In [ ]:
# sort based on rows or columns
df0.set_index('region').reset_index().sort_index(axis=1).head(3)

,firm,profit,region,sales,year
0,A,15,East,120.0,2020
1,A,18,East,135.0,2021
2,A,18,East,135.0,2022


In [56]:
lt10 = df0.sales.isna()
df0.loc[lt10 & (df0.profit >15)].head(3)

,firm,year,region,sales,profit
id-9,C,2021,East,NaN,16


In [ ]:
df0.query('(profit >15) &  @lt10').head(3)

,firm,year,region,sales,profit
id-9,C,2021,East,NaN,16


## Indexing by Position
| Method | Description |
|---|---|
| `.iloc` | Attribute to index off of by index and column positions. Slices use the half-open interval (including start but not end). |

## Indexing by Name
| Method | Description |
|---|---|
| `.loc` | Attribute to index off of by index and column names. Slices use the closed interval (including start and end). |

Note: to work with the index name, I can
- slice rows based on the initial of the index


In [ ]:
# If you convert the categorical index to a string index, then you can use partial strings:
region_order = pd.CategoricalDtype(['West','South','East', 'North'], ordered=True)

(df0
 .assign(region=df0.region.astype(region_order))
 .set_index('region')
 .sort_index()
 .loc[lambda d: d.index>'South']
)

,firm,year,sales,profit
region,,,,
East,A,2020,120.0,15
East,A,2021,135.0,18
East,A,2022,135.0,18
East,A,2023,160.0,25
East,C,2020,120.0,15
East,C,2021,NaN,16
East,C,2022,130.0,19
East,C,2023,130.0,19


## Filtering with Functions & .loc

- `.where(cond, other=nan, axis=None, level=None, errors='raise', try_cast=None)`
    - keeps values `True`, and replaces values `False`
    - only 2 arguments!
- `.fillna(value=, method=None, axis=None, limit=None, downcast=None)`: Return a DataFrame with missing values filled.
- `np.where(condition, x=None, y=None)`: keeps values `True`, and replaces values `False`.
    - 3 arguments